# Kasus B: Sentimen Ulasan Produk Video Games

Target metrik: **macro-F1** (rata-rata F1 kelas negatif dan positif).

Data: Amazon Reviews 2023, kategori `Video_Games` (`data/Video_Games.jsonl.gz`), hanya ulasan dengan timestamp sampai 31 Desember 2019.
Label: rating 1-2 = negatif (0), rating 4-5 = positif (1), rating 3 dibuang.

In [1]:
import gzip
import json
import re
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "Video_Games.jsonl.gz"
URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz"
MODEL_PATH = ROOT / "models" / "kasus_b_sentimen_tfidf_logreg.joblib"
SEED = 42
MAX_CHARS = 2000
TRAIN_SAMPLE = 1_000_000

## 1. Baca file per baris

File ~800 MB terkompresi (~4,6 juta ulasan), jadi tidak dimuat utuh. Setiap baris dibaca satu per satu dan hanya kolom yang dibutuhkan yang disimpan: judul, teks, rating, tahun.

In [2]:
if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, DATA)

CUTOFF_MS = int(datetime(2020, 1, 1, tzinfo=timezone.utc).timestamp() * 1000)
TAG = re.compile(r"<[^>]+>")

rows = []
with gzip.open(DATA, "rt", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        rating = int(r["rating"])
        if r["timestamp"] >= CUTOFF_MS or rating == 3:
            continue
        text = TAG.sub(" ", f"{r.get('title', '')}. {r.get('text', '')}")[:MAX_CHARS]
        year = datetime.fromtimestamp(r["timestamp"] / 1000, timezone.utc).year
        rows.append((text, int(rating >= 4), year))

df = pd.DataFrame(rows, columns=["text", "label", "year"])
del rows
print(df.shape)
df.groupby("year")["label"].agg(["size", "mean"]).tail(8)

(2813797, 3)


,size,mean
year,,
2012,101946,0.817786
2013,228417,0.832915
2014,297986,0.849902
2015,381395,0.846922
2016,362422,0.831569
2017,338738,0.812820
2018,354563,0.798730
2019,421794,0.824545


## 2. Pembagian data: berdasarkan waktu

- **Latih**: ulasan sampai 2018. **Validasi**: semua ulasan tahun 2019.
- Alasan: model akan dipakai untuk ulasan *baru*. Split per waktu meniru kondisi itu dan menangkap perubahan bahasa/produk dari tahun ke tahun; split acak membuat skor terlalu optimis karena ulasan dari periode yang sama (bahkan produk yang sama) ada di latih dan validasi.
- Data latih (~2,5 juta) diambil sampel acak 1 juta ulasan agar muat di memori laptop (16 GB). Proporsi kelas tetap seperti aslinya.
- Kelas tidak seimbang (~80% positif), jadi model memakai `class_weight="balanced"` agar kelas negatif tidak diabaikan.

In [3]:
train_all = df[df["year"] <= 2018]
valid = df[df["year"] == 2019]
train = train_all.sample(n=min(TRAIN_SAMPLE, len(train_all)), random_state=SEED)
del df, train_all
print("latih:", train.shape, f"positif {train['label'].mean():.3f}")
print("validasi:", valid.shape, f"positif {valid['label'].mean():.3f}")

latih: (1000000, 3) positif 0.824


validasi: (421794, 3) positif 0.825


## 3. Model: TF-IDF + Logistic Regression

Alasan memilih:
- Baseline kuat untuk klasifikasi sentimen; kata dan frasa dua kata (*"not worth"*, *"waste of"*) sudah sangat informatif.
- Dilatih dalam hitungan menit di CPU untuk jutaan teks, tanpa GPU. Model transformer (mis. BERT) bisa sedikit lebih akurat, tapi butuh GPU dan waktu jauh lebih lama.
- Mudah dijelaskan: bobot tiap kata bisa dilihat langsung.
- `class_weight="balanced"` menyamakan pengaruh kedua kelas, sesuai tujuan macro-F1.

In [4]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9,
        max_features=300_000,
        sublinear_tf=True,
        dtype=np.float32,
    )),
    ("clf", LogisticRegression(C=4.0, solver="liblinear", class_weight="balanced", max_iter=1000)),
])
model.fit(train["text"], train["label"])

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(dtype=<class 'numpy.float32'>, max_df=0.9,
                                 max_features=300000, min_df=5,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('clf',
                 LogisticRegression(C=4.0, class_weight='balanced',
                                    max_iter=1000, solver='liblinear'))])

In [5]:
valid_pred = model.predict(valid["text"])
macro_f1 = f1_score(valid["label"], valid_pred, average="macro")
print(f"Validasi (2019) macro-F1: {macro_f1:.4f}\n")
print(classification_report(valid["label"], valid_pred, target_names=["negatif", "positif"], digits=4))

Validasi (2019) macro-F1: 0.9393

              precision    recall  f1-score   support

     negatif     0.8545    0.9530    0.9011     74006
     positif     0.9898    0.9655    0.9775    347788

    accuracy                         0.9633    421794
   macro avg     0.9221    0.9592    0.9393    421794
weighted avg     0.9660    0.9633    0.9641    421794



## 4. Kata paling berpengaruh

In [6]:
vocab = model.named_steps["tfidf"].get_feature_names_out()
coef = model.named_steps["clf"].coef_[0]
order = np.argsort(coef)
pd.DataFrame({"paling negatif": vocab[order[:15]], "paling positif": vocab[order[::-1][:15]]})

,paling negatif,paling positif
0,two stars,great
1,one star,five stars
2,not worth,awesome
3,not,four stars
4,worst,perfect
5,disappointing,best
6,terrible,not bad
7,disappointed,amazing
8,disappointment,good
9,awful,excellent


## 5. Simpan model lalu muat ulang

In [7]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(model, MODEL_PATH, compress=3)

reloaded = joblib.load(MODEL_PATH)
assert (reloaded.predict(valid["text"]) == valid_pred).all()
print("Model dimuat ulang dari", MODEL_PATH.relative_to(ROOT))

contoh = [
    "Great game. My kids play it every day, totally worth the price.",
    "Broken disc. Stopped working after two days, waste of money.",
]
for teks, p in zip(contoh, reloaded.predict(contoh)):
    print(["negatif", "positif"][p], "<-", teks)

Model dimuat ulang dari models/kasus_b_sentimen_tfidf_logreg.joblib
positif <- Great game. My kids play it every day, totally worth the price.
negatif <- Broken disc. Stopped working after two days, waste of money.
